In [ ]:
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from const import API_key
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

In [ ]:
url = "http://api.openweathermap.org/data/2.5/weather"
params = {
    "appid": API_key,
    "units": "metric"  
}

In [ ]:
iran_province_capitals = {
    "Alborz": "Karaj",
    "Ardebil": "Ardabil",
    "Bushehr": "Bushehr",
    "ChaharMahallandBakhtiari": "Shahr-e Kord",
    "EastAzarbaijan": "Tabriz",
    "Esfahan": "Esfahan",
    "Fars": "Shiraz",
    "Gilan": "Rasht",
    "Golestan": "Gorgan",
    "Hamadan": "Hamadan",
    "Hormozgan": "Bandar Abbas",
    "Ilam": "Ilam",
    "Isfahan": "Isfahan",
    "Kerman": "Kerman",                       
    "Kermanshah": "Kermanshah",
    "NorthKhorasan": "Bojnurd",
    "RazaviKhorasan": "Mashhad",
    "SouthKhorasan": "Birjand",
    "Khuzestan": "Ahvaz",
    "KohgiluyehandBuyerAhmad": "Yasuj",
    "Kordestan": "Sanandaj",
    "Lorestan": "Khorramabad",
    "Markazi": "Arak",
    "Mazandaran": "Sari",
    "Qazvin": "Qazvin",
    "Qom": "Qom",
    "Semnan": "Semnan",
    "SistanandBaluchestan": "Zahedan",
    "Tehran": "Tehran",
    "WestAzarbaijan": "Urmia",
    "Yazd": "Yazd",
    "Zanjan": "Zanjan"
}


In [ ]:
weather_data = pd.DataFrame(columns=['province', 'temp', 'humidity', 'description'])

for province, city in iran_province_capitals.items():
    params['q'] = city
    response = requests.get(url, params=params)
    data = response.json()
    if response.status_code == 200:
        new_row = pd.DataFrame({
            'province': [province],
            'temp': [data['main']['temp']],
            'humidity' : [data['main']['humidity']],
            'description' : [data['weather'][0]['description']]
        })
        weather_data = pd.concat([weather_data, new_row])
        
    else:
        print(f"Failed to fetch data for {city}")



In [ ]:
weather_data = weather_data.rename(columns={'province': 'NAME_1'})

In [ ]:
# Load the GeoJSON file
gdf = gpd.read_file('geojson/gadm41_IRN_1.json')

province_data  = pd.merge(gdf, weather_data, on='NAME_1')

In [ ]:
# Plot the map
fig = px.choropleth(province_data,
                    geojson=province_data.geometry,
                    locations=province_data.index,
                    color=province_data.temp, 
                    hover_name=province_data['NAME_1'],  
                    title="Iran and Its Provinces",
                    hover_data=['humidity', 'temp', 'description'])

fig.update_geos(fitbounds="locations", visible=True)

fig.show()